# SQL Baseline — PartnerLens

**PartnerLens Copilot — Final Submission Notebook**

This notebook uses repository-relative paths and does not require Google Drive. Run it from a cloned copy of the `partnerlens-copilot` repository after installing `requirements.txt`.

## Environment setup

From a terminal opened at the repository root, install dependencies once:

```bash
python -m pip install -r requirements.txt
```

The next cell locates the repository automatically when Jupyter is started from the repository or its `notebooks` folder.

In [1]:
from pathlib import Path
import importlib.util
import os
import sys


def locate_repository_root() -> Path:
    """Locate the PartnerLens repository without relying on Google Drive."""
    current = Path.cwd().resolve()
    candidates = [
        current,
        *current.parents,
        Path.home() / "partnerlens-copilot",
        Path("/content/partnerlens-copilot"),
    ]

    visited = set()
    for candidate in candidates:
        try:
            candidate = candidate.expanduser().resolve()
        except OSError:
            continue
        if candidate in visited:
            continue
        visited.add(candidate)
        if (
            (candidate / "README.md").is_file()
            and (candidate / "src").is_dir()
            and (candidate / "data").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "PartnerLens repository root was not found. Start Jupyter from inside "
        "the partnerlens-copilot repository, or clone the repository to "
        "~/partnerlens-copilot or /content/partnerlens-copilot."
    )


REPO_ROOT = locate_repository_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

required_modules = {
    "pandas": "pandas",
    "numpy": "numpy",
    "sqlparse": "sqlparse",
    "openpyxl": "openpyxl",
}
missing_packages = [
    package_name
    for module_name, package_name in required_modules.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_packages:
    raise ModuleNotFoundError(
        "Missing required packages: "
        + ", ".join(missing_packages)
        + ". From the repository root run: "
        + f"{sys.executable} -m pip install -r requirements.txt"
    )

RAW_DIR = REPO_ROOT / "data" / "raw"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
CONFIG_DIR = REPO_ROOT / "configs"
NOTEBOOK_OUTPUT_DIR = REPO_ROOT / "artifacts" / "notebook_outputs"
EVALUATION_OUTPUT_DIR = REPO_ROOT / "artifacts" / "evaluation"
NOTEBOOK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EVALUATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {REPO_ROOT}")
print(f"Python executable: {sys.executable}")
print("Notebook environment is ready.")

Repository root: C:\Users\nafee\partnerlens-copilot
Python executable: C:\Users\nafee\AppData\Local\Programs\Python\Python312\python.exe
Notebook environment is ready.


## 1. Build or locate the SQLite database

The database is generated from the processed CSV files using the same `src.database_setup` module used by the application. The canonical table names are preserved across the notebooks and source modules.

In [2]:
import sqlite3
import pandas as pd
from IPython.display import display

from src.database_setup import DATABASE_PATH, create_database
from src.query_executor import execute_query_detailed
from src.sql_validator import validate_sql_detailed

if not DATABASE_PATH.is_file():
    print("partnerlens.db was not found; building it from processed CSV files.")
    create_database()
else:
    print(f"Using existing database: {DATABASE_PATH.relative_to(REPO_ROOT)}")

Using existing database: data\processed\partnerlens.db


In [3]:
with sqlite3.connect(DATABASE_PATH) as connection:
    database_tables = pd.read_sql_query(
        """
        SELECT name AS table_name
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name
        """,
        connection,
    )

    row_count_rows = []
    for table_name in database_tables["table_name"]:
        safe_table_name = table_name.replace('"', '""')
        row_count = connection.execute(
            f'SELECT COUNT(*) FROM "{safe_table_name}"'
        ).fetchone()[0]
        row_count_rows.append({"table_name": table_name, "row_count": row_count})

row_counts_df = pd.DataFrame(row_count_rows)
display(row_counts_df)

canonical_tables = {
    "partners",
    "partner_pricing",
    "monthly_partner_metrics",
    "partner_current_preview",
}
missing_canonical_tables = sorted(canonical_tables - set(row_counts_df["table_name"]))
if missing_canonical_tables:
    raise AssertionError("Missing canonical SQLite tables: " + ", ".join(missing_canonical_tables))

print("All canonical PartnerLens SQLite tables are present.")

,table_name,row_count
0,monthly_partner_metrics,120000
1,partner_current_preview,1000
2,partner_pricing,5000
3,partners,5000


All canonical PartnerLens SQLite tables are present.


## 2. Baseline SQL cases

Every query uses explicit columns, approved tables, a single read-only `SELECT`, and a bounded `LIMIT`. These are the same controls enforced by `src/sql_validator.py`.

In [4]:
baseline_cases = [
    {
        "case_id": "AZ_GROWTH",
        "description": "Arizona partners with transaction growth above 20%",
        "sql": """
            SELECT partner_id, partner_name, industry_vertical, state,
                   txn_growth_pct, payment_volume_usd, net_revenue_usd
            FROM partner_current_preview
            WHERE state = 'AZ' AND txn_growth_pct > 20
            ORDER BY txn_growth_pct DESC, partner_name ASC
            LIMIT 25;
        """,
    },
    {
        "case_id": "TOP_VOLUME",
        "description": "Top partners by payment volume",
        "sql": """
            SELECT partner_id, partner_name, industry_vertical, state,
                   payment_volume_usd, txn_count, net_revenue_usd
            FROM partner_current_preview
            ORDER BY payment_volume_usd DESC, partner_name ASC
            LIMIT 10;
        """,
    },
    {
        "case_id": "PARTNER_RISK",
        "description": "Partner risk and compliance directory",
        "sql": """
            SELECT partner_id, partner_name, industry_vertical, state,
                   risk_tier, kyc_status, pci_level
            FROM partners
            ORDER BY partner_name ASC
            LIMIT 25;
        """,
    },
    {
        "case_id": "PARTNER_PRICING",
        "description": "Partner pricing assignments with partner names",
        "sql": """
            SELECT pp.partner_id, p.partner_name, p.industry_vertical, p.state,
                   pp.pricing_plan_id, pp.recommended_pricing_plan_id,
                   pp.negotiated_bps, pp.negotiated_per_txn_fee_usd,
                   pp.exception_flag, pp.approval_status
            FROM partner_pricing AS pp
            LEFT JOIN partners AS p ON pp.partner_id = p.partner_id
            ORDER BY pp.exception_flag DESC, p.partner_name ASC
            LIMIT 25;
        """,
    },
    {
        "case_id": "PARTNER_DIRECTORY",
        "description": "General partner directory",
        "sql": """
            SELECT partner_id, partner_name, partner_type, industry_vertical,
                   partner_size, partner_status, state, region, risk_tier, kyc_status
            FROM partners
            ORDER BY partner_name ASC
            LIMIT 25;
        """,
    },
]

baseline_results = []
baseline_outputs = {}
for test_case in baseline_cases:
    validation = validate_sql_detailed(test_case["sql"])
    execution_status = "not_run"
    row_count = 0
    elapsed_ms = None
    output_columns = []

    if validation.is_valid:
        execution = execute_query_detailed(test_case["sql"], db_path=DATABASE_PATH)
        execution_status = execution.status
        row_count = execution.row_count
        elapsed_ms = execution.elapsed_ms
        output_columns = list(execution.columns)
        baseline_outputs[test_case["case_id"]] = execution.dataframe

    baseline_results.append(
        {
            "case_id": test_case["case_id"],
            "description": test_case["description"],
            "validation_passed": validation.is_valid,
            "validation_reason_code": validation.reason_code,
            "referenced_tables": ", ".join(validation.referenced_tables),
            "validated_limit": validation.row_limit,
            "execution_status": execution_status,
            "row_count": row_count,
            "elapsed_ms": elapsed_ms,
            "output_columns": ", ".join(output_columns),
        }
    )

baseline_results_df = pd.DataFrame(baseline_results)
display(baseline_results_df)

,case_id,description,validation_passed,validation_reason_code,referenced_tables,validated_limit,execution_status,row_count,elapsed_ms,output_columns
0,AZ_GROWTH,Arizona partners with transaction growth above...,True,validation_passed,partner_current_preview,25,success,14,0.0,"partner_id, partner_name, industry_vertical, s..."
1,TOP_VOLUME,Top partners by payment volume,True,validation_passed,partner_current_preview,10,success,10,0.0,"partner_id, partner_name, industry_vertical, s..."
2,PARTNER_RISK,Partner risk and compliance directory,True,validation_passed,partners,25,success,25,16.0,"partner_id, partner_name, industry_vertical, s..."
3,PARTNER_PRICING,Partner pricing assignments with partner names,True,validation_passed,"partner_pricing, partners",25,success,25,31.0,"partner_id, partner_name, industry_vertical, s..."
4,PARTNER_DIRECTORY,General partner directory,True,validation_passed,partners,25,success,25,16.0,"partner_id, partner_name, partner_type, indust..."


In [5]:
for case_id, dataframe in baseline_outputs.items():
    print(f"\n{case_id}: showing up to five rows")
    display(dataframe.head(5))


AZ_GROWTH: showing up to five rows


,partner_id,partner_name,industry_vertical,state,txn_growth_pct,payment_volume_usd,net_revenue_usd
0,P000011,Maple Checkout 0011,Hospitality,AZ,48.51,14054.99,231.40
1,P000564,Keystone Connect 0564,Logistics,AZ,42.51,1256911.71,9303.17
2,P000473,Cedar Solutions 0473,Retail,AZ,41.67,210078.63,1743.91
3,P000286,Keystone Payments 0286,Travel,AZ,39.94,342631.51,6862.54
4,P000515,Brightpath Platform 0515,Hospitality,AZ,35.77,13782636.25,53500.69



TOP_VOLUME: showing up to five rows


,partner_id,partner_name,industry_vertical,state,payment_volume_usd,txn_count,net_revenue_usd
0,P000610,Skyline Payments 0610,Gaming,WA,3.009404e+08,11100654,4608699.76
1,P000645,Silverline Processing 0645,Logistics,CA,1.229815e+08,394133,311941.07
2,P000969,Summit Gateway 0969,Retail,GA,1.213787e+08,1795726,750140.84
3,P000340,Summit Financial 0340,Government,CA,1.000062e+08,444622,311878.69
4,P000760,BluePeak Billing 0760,Utilities,PA,9.948762e+07,1035520,299584.68



PARTNER_RISK: showing up to five rows


,partner_id,partner_name,industry_vertical,state,risk_tier,kyc_status,pci_level
0,P000335,BluePeak Billing 0335,Gaming,AZ,Medium,Approved,Level 2
1,P000573,BluePeak Billing 0573,Utilities,NY,Low,Approved,Level 2
2,P000662,BluePeak Billing 0662,SaaS,MN,Medium,Expired,Level 2
3,P000735,BluePeak Billing 0735,Retail,NY,Low,Approved,Level 1
4,P000760,BluePeak Billing 0760,Utilities,PA,Low,Expired,Level 4



PARTNER_PRICING: showing up to five rows


,partner_id,partner_name,industry_vertical,state,pricing_plan_id,recommended_pricing_plan_id,negotiated_bps,negotiated_per_txn_fee_usd,exception_flag,approval_status
0,P000573,BluePeak Billing 0573,Utilities,NY,LEGACY_STANDARD,LAUNCH,53.65,0.0601,1,Approved
1,P000735,BluePeak Billing 0735,Retail,NY,PLATFORM_ISV,GROWTH,55.48,0.0599,1,Approved
2,P004415,BluePeak Billing 4415,SaaS,NY,GROWTH,PLATFORM_ISV,52.99,0.0635,1,Pending
3,P002322,BluePeak Checkout 2322,Hospitality,CO,LEGACY_STANDARD,LAUNCH,67.65,0.0541,1,Pending
4,P002996,BluePeak Checkout 2996,Retail,NJ,SCALE,SCALE,48.18,0.0429,1,Approved



PARTNER_DIRECTORY: showing up to five rows


,partner_id,partner_name,partner_type,industry_vertical,partner_size,partner_status,state,region,risk_tier,kyc_status
0,P000335,BluePeak Billing 0335,ISO/Reseller,Gaming,Strategic,Active,AZ,West,Medium,Approved
1,P000573,BluePeak Billing 0573,Marketplace,Utilities,SMB,Active,NY,Northeast,Low,Approved
2,P000662,BluePeak Billing 0662,ISO/Reseller,SaaS,SMB,Active,MN,Midwest,Medium,Expired
3,P000735,BluePeak Billing 0735,ISO/Reseller,Retail,Mid-Market,Active,NY,Northeast,Low,Approved
4,P000760,BluePeak Billing 0760,Merchant Acquirer,Utilities,Strategic,Active,PA,Northeast,Low,Expired


## 3. SQL baseline metrics

These metrics are saved under `artifacts/notebook_outputs/` so the final repository exposes validation and execution evidence directly.

In [6]:
sql_metrics_df = pd.DataFrame(
    [
        {
            "metric": "baseline_case_count",
            "value": len(baseline_results_df),
        },
        {
            "metric": "sql_validation_pass_rate_pct",
            "value": round(100 * baseline_results_df["validation_passed"].mean(), 2),
        },
        {
            "metric": "query_execution_success_rate_pct",
            "value": round(
                100 * baseline_results_df["execution_status"].eq("success").mean(),
                2,
            ),
        },
        {
            "metric": "average_execution_time_ms",
            "value": round(
                baseline_results_df["elapsed_ms"].dropna().astype(float).mean(),
                2,
            ),
        },
        {
            "metric": "canonical_table_presence_pct",
            "value": round(
                100 * len(canonical_tables & set(row_counts_df["table_name"])) / len(canonical_tables),
                2,
            ),
        },
    ]
)

results_path = NOTEBOOK_OUTPUT_DIR / "sql_baseline_case_results.csv"
metrics_path = NOTEBOOK_OUTPUT_DIR / "sql_baseline_metrics.csv"
tables_path = NOTEBOOK_OUTPUT_DIR / "sqlite_table_row_counts.csv"

baseline_results_df.to_csv(results_path, index=False)
sql_metrics_df.to_csv(metrics_path, index=False)
row_counts_df.to_csv(tables_path, index=False)

for output_path in (results_path, metrics_path, tables_path):
    print(f"Saved: {output_path.relative_to(REPO_ROOT)}")

display(sql_metrics_df)

if not baseline_results_df["validation_passed"].all():
    raise AssertionError("At least one baseline SQL query failed validation.")
if not baseline_results_df["execution_status"].eq("success").all():
    raise AssertionError("At least one baseline SQL query failed execution.")

print("All SQL baseline cases passed validation and executed successfully.")

Saved: artifacts\notebook_outputs\sql_baseline_case_results.csv
Saved: artifacts\notebook_outputs\sql_baseline_metrics.csv
Saved: artifacts\notebook_outputs\sqlite_table_row_counts.csv


,metric,value
0,baseline_case_count,5.0
1,sql_validation_pass_rate_pct,100.0
2,query_execution_success_rate_pct,100.0
3,average_execution_time_ms,12.6
4,canonical_table_presence_pct,100.0


All SQL baseline cases passed validation and executed successfully.


## Final interpretation

A successful run confirms that the processed data can be reproduced as SQLite tables and that the controlled baseline queries are valid, bounded, read-only, and executable. The metrics CSV files provide visible repository evidence for the final evaluation.